<div style="background: linear-gradient(135deg, rgba(40,167,69,0.18) 0%, rgba(34,139,34,0.12) 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(40,167,69,0.3);
            border-left: 6px solid #28a745;
            padding: 30px;
            border-radius: 12px;
            margin: 25px 0;
            box-shadow: 0 8px 20px rgba(40,167,69,0.25);">
    <h1 style="margin: 0 0 15px 0; color: #ffffff; font-size: 2em; font-weight: 700;">
        ✅ SOLUTIONS - Part 2: Context Management & Custom Tools
    </h1>
    <h3 style="margin: 0 0 20px 0; color: #ffffff; opacity: 0.95; font-weight: 400; font-size: 1.15em;">
        Complete Reference Implementation
    </h3>
    <div style="background: rgba(255,255,255,0.08);
                backdrop-filter: blur(8px);
                border: 1px solid rgba(255,255,255,0.15);
                padding: 20px;
                border-radius: 8px;
                margin-top: 20px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.15);">
        <p style="margin: 0 0 12px 0; font-size: 1.1em; color: #ffffff;">
            <strong style="color: #ffffff;">🚀 Quick Start:</strong> To see the complete implementation:
        </p>
        <ol style="margin: 10px 0; padding-left: 25px; line-height: 1.8; font-size: 1.05em; color: #ffffff;">
            <li><strong>Select Kernel (Python 3.13) → Menu → Run All</strong> (or press Shift+Enter through each cell)</li>
            <li>Watch the notebook execute all setup and solution code</li>
            <li>Review the get_trending_products() solution with explanatory comments</li>
        </ol>
        <p style="margin: 15px 0 0 0; font-size: 1.02em; color: #ffffff;">
            ⏱️ <strong style="color: #ffffff;">Total Runtime:</strong> ~1-2 minutes
        </p>
    </div>
</div>

---

## ⚙️ Step 0: Select Python Kernel

<div style="background: linear-gradient(135deg, rgba(245,158,11,0.18) 0%, rgba(217,119,6,0.18) 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(245,158,11,0.35);
            border-left: 6px solid #f59e0b;
            padding: 20px;
            border-radius: 10px;
            margin: 20px 0;
            box-shadow: 0 4px 16px rgba(245,158,11,0.25);">
    <h4 style="margin: 0 0 15px 0; color: #fbbf24; font-size: 1.2em; font-weight: 600;">
        ⚠️ IMPORTANT FIRST STEP
    </h4>
    <p style="margin: 0; color: #e9ecef; line-height: 1.6; font-size: 1.02em;">
        Before running any code, ensure you're using <strong style="color: #fbbf24;">Python 3.13</strong> kernel.
    </p>
</div>

In [ ]:
# Verify Python Environment
import sys

python_version = sys.version.split()[0]
required_version = "3.13"

print(f"🐍 Python Version: {python_version}")
print("─" * 50)

if python_version.startswith(required_version):
    print("✅ Environment Verified")
    print(f"   Running Python {python_version} (Required: {required_version}.x)")
    print("   Ready to proceed with workshop exercises.")
else:
    print(f"⚠️  Environment Mismatch Detected")
    print(f"   Current: Python {python_version}")
    print(f"   Required: Python {required_version}.x")
    print("\n📋 Action Required:")
    print("   1. Click the kernel selector (top-right corner)")
    print("   2. Select 'Python 3.13' from the dropdown")
    print("   3. Re-run this cell to verify")

---

# Setup: Database Connection and Bedrock Client

In [ ]:
# ============================================================================
# Environment Setup: Database Connection & Bedrock Client Initialization
# ============================================================================

import os
import json
import psycopg
from psycopg.rows import dict_row
import boto3
from typing import List, Dict, Optional
from dotenv import load_dotenv

# Load environment configuration
load_dotenv('/workshop/sample-dat406-build-agentic-ai-powered-search-apg/.env')

# ============================================================================
# Database Configuration
# ============================================================================

DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'postgres')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

# Construct PostgreSQL connection string
conn_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# ============================================================================
# Amazon Bedrock Client Initialization
# ============================================================================

bedrock = boto3.client(
    'bedrock-runtime', 
    region_name=os.environ.get('AWS_REGION', 'us-west-2')
)

# ============================================================================
# Verification
# ============================================================================

print("=" * 60)
print("✅ Environment Configuration Complete")
print("=" * 60)
print(f"📊 Aurora Endpoint: {conn_string.split('@')[1].split('/')[0]}")
print(f"🤖 Bedrock Region:  {bedrock.meta.region_name}")
print(f"🗄️  Database Name:  {DB_NAME}")
print("=" * 60)

In [ ]:
def generate_embedding(text: str) -> List[float]:
    """
    Generate semantic embeddings using Amazon Titan Text Embeddings V2.
    
    Args:
        text: Input text to embed
        
    Returns:
        List of 1024 floating-point values representing the semantic embedding
    """
    response = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({"inputText": text})
    )
    
    result = json.loads(response['body'].read())
    return result['embedding']

print("✅ Embedding generation function ready")

---

# 2C. TODO Exercise Solution: get_trending_products()

<div style="background: linear-gradient(135deg, rgba(40,167,69,0.18) 0%, rgba(34,139,34,0.12) 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(40,167,69,0.3);
            border-left: 6px solid #28a745;
            padding: 25px;
            border-radius: 12px;
            margin: 20px 0;
            box-shadow: 0 8px 20px rgba(40,167,69,0.25);">
    <h2 style="margin: 0 0 15px 0; color: #ffffff; font-weight: 600;">
        ✅ Complete Implementation
    </h2>
    <p style="margin: 0; color: #ffffff; font-size: 1.1em; line-height: 1.6; opacity: 0.95;">
        Here's the complete solution for building get_trending_products() from Section 2C.
    </p>
</div>

In [ ]:
from strands import tool

# ============================================================================
# SOLUTION: get_trending_products() - Complete Implementation
# ============================================================================

@tool
def get_trending_products(limit: int = 5) -> str:
    """
    Get trending products based on ratings and reviews.
    Returns products with high ratings (4.5+) and many reviews (50+).
    
    Args:
        limit: Maximum number of products to return (default: 5)
    
    Returns:
        JSON string with trending products
    """
    try:
        # SOLUTION: Connect to database with dict_row factory for dictionary results
        with psycopg.connect(conn_string) as conn:
            with conn.cursor(row_factory=dict_row) as cur:
                
                # SOLUTION: Build SELECT query with THREE filters in WHERE clause
                # 1. stars >= 4.5 (high ratings)
                # 2. reviews > 50 (proven popularity)
                # 3. quantity > 0 (in stock)
                # ORDER BY stars DESC, reviews DESC (best ratings first, then popularity)
                cur.execute("""
                    SELECT 
                        "productId", 
                        product_description, 
                        category_name, 
                        price, 
                        stars, 
                        reviews,
                        quantity, 
                        "imgUrl", 
                        "productURL"
                    FROM bedrock_integration.product_catalog
                    WHERE stars >= 4.5 
                      AND reviews > 50 
                      AND quantity > 0
                    ORDER BY stars DESC, reviews DESC
                    LIMIT %s
                """, (limit,))
                
                products = cur.fetchall()
                
                # SOLUTION: Format results as structured JSON
                # Include: trending_products array, count, and criteria used
                result = {
                    "trending_products": [
                        {
                            "id": p["productId"],
                            "title": p["product_description"],
                            "category": p["category_name"],
                            "price": float(p["price"]),
                            "stars": float(p["stars"]),
                            "reviews": p["reviews"],
                            "imgUrl": p.get("imgUrl", ""),
                            "productURL": p.get("productURL", "#")
                        }
                        for p in products
                    ],
                    "count": len(products),
                    "criteria": "stars >= 4.5, reviews > 50, in stock"
                }
                
                return json.dumps(result, indent=2)
                
    except Exception as e:
        return json.dumps({"error": f"Database error: {str(e)}"})

print("✅ Solution function defined: get_trending_products()")

## Testing the Solution

Let's verify the implementation works correctly.

In [ ]:
# ============================================================================
# Verification: Test Your Implementation
# ============================================================================

print("🧪 Testing get_trending_products()...\n")

try:
    result = get_trending_products(limit=3)
    data = json.loads(result)
    
    if "trending_products" in data and len(data["trending_products"]) > 0:
        print("✅ Tool working correctly!\n")
        print("🔥 Trending Products:")
        print("-" * 60)
        
        for i, product in enumerate(data["trending_products"], 1):
            print(f"{i}. {product['title'][:50]}")
            print(f"   ${product['price']:.2f} | {product['stars']}⭐ | {product['reviews']:,} reviews\n")
        
        print(f"Criteria: {data['criteria']}")
        print(f"Results: {data['count']} products")
        
        # Verify business logic
        all_high_rated = all(p['stars'] >= 4.5 for p in data["trending_products"])
        all_popular = all(p['reviews'] > 50 for p in data["trending_products"])
        
        if all_high_rated and all_popular:
            print("\n✅ Business logic verified!")
        else:
            print("\n⚠️  Check WHERE clause - some products don't meet criteria")
            
    else:
        print("❌ No results. Check your implementation.")
        
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {e}")

print("\n" + "=" * 60)
print("🎉 If test passed, you've successfully built:")
print("   • Custom tool with @tool decorator")
print("   • Business logic with THREE filter conditions")
print("   • Structured JSON response")
print("   • Production-ready implementation")
print("=" * 60)

<div style="background: linear-gradient(135deg, rgba(40,167,69,0.18) 0%, rgba(34,139,34,0.12) 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(40,167,69,0.3);
            padding: 20px;
            border-radius: 10px;
            margin: 30px 0;
            text-align: center;
            box-shadow: 0 6px 16px rgba(40,167,69,0.25);">
    <h3 style="margin: 0 0 10px 0; color: #ffffff; font-size: 1.4em; font-weight: 600;">
        🎉 Solution Complete!
    </h3>
    <p style="margin: 0; color: #ffffff; font-size: 1.05em; line-height: 1.6; opacity: 0.95;">
        You've seen the complete implementation of get_trending_products().<br>
        <strong style="color: #ffffff;">Ready for Part 3?</strong> Build multi-agent orchestration!
    </p>
</div>

---

<div style="text-align: center; color: #888; font-size: 0.9em; margin-top: 30px;">
Built for AWS re:Invent 2025 | DAT406 Workshop
</div>